In [5]:

import pandas as pd
from pathlib import Path

# ============================================================
# DAY 10 - STEP 2: CUSTOMER FEATURE AND TARGET DEFINITIONS
# ============================================================

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

# ------------------------------------------------------------
# 1. CUSTOMER TARGET DEFINITIONS
# ------------------------------------------------------------

target_definitions = pd.DataFrame([
    {
        "Target": "Inactive_180D",
        "Definition": (
            "1 if no quotation is recorded in the "
            "180 days before the assessment date. "
            "This is a provisional inactivity proxy, "
            "not a verified churn outcome."
        ),
        "Required_Fields": (
            "Company_ID, Quote_Date, Assessment_Date"
        ),
        "Status": "Provisional synthetic proxy",
        "Approval": "Business confirmation pending"
    },
    {
        "Target": "Customer_Segment",
        "Definition": (
            "Rule-based descriptive grouping using quotation "
            "recency, quoted value, and inactivity. Segment "
            "priority: Inactive Quote, High Quote Value, "
            "Recent Activity, then Standard / Review."
        ),
        "Required_Fields": (
            "Company_ID, Quote_ID, Quote_Value, "
            "Recency_Days, Inactive_180D_Proxy"
        ),
        "Status": "Demonstration segmentation",
        "Approval": "Business confirmation pending"
    },
    {
        "Target": "Customer_Lifetime_Value",
        "Definition": (
            "Forward-looking customer value estimate. "
            "Actual collected revenue, delivery costs, "
            "and retention history are required for "
            "a validated estimate. Quoted value is not "
            "actual collected revenue."
        ),
        "Required_Fields": (
            "Customer_ID, Transaction_Date, "
            "Collected_Revenue, Delivery_Cost, "
            "Retention_History"
        ),
        "Status": "Actual inputs unavailable",
        "Approval": "Finance confirmation pending"
    }
])

# ------------------------------------------------------------
# 2. CUSTOMER FEATURE DICTIONARY
# ------------------------------------------------------------

feature_dictionary = pd.DataFrame([
    {
        "Feature": "Company_ID",
        "Definition": (
            "Identifier linking quotations to a company"
        ),
        "Source": "Synthetic quotation dataset",
        "Feature_Type": "Identifier",
        "Use_In_Model": "No"
    },
    {
        "Feature": "Quote_Count",
        "Definition": (
            "Number of recorded quotations per company"
        ),
        "Source": "Quote_ID",
        "Feature_Type": "Frequency",
        "Use_In_Model": "Potentially"
    },
    {
        "Feature": "Total_Quoted_Value",
        "Definition": (
            "Sum of quotation values per company; "
            "not actual collected revenue"
        ),
        "Source": "Quote_Value",
        "Feature_Type": "Monetary proxy",
        "Use_In_Model": "Potentially"
    },
    {
        "Feature": "Last_Quote_Date",
        "Definition": (
            "Most recent recorded quotation date"
        ),
        "Source": "Quote_Date",
        "Feature_Type": "Recency",
        "Use_In_Model": "Potentially"
    },
    {
        "Feature": "Recency_Days",
        "Definition": (
            "Days from the last quotation to the "
            "assessment date"
        ),
        "Source": "Quote_Date and Assessment_Date",
        "Feature_Type": "Recency",
        "Use_In_Model": "Potentially"
    },
    {
        "Feature": "Inactive_180D",
        "Definition": (
            "Provisional indicator that no quotation "
            "was recorded in the preceding 180 days; "
            "not a verified churn label"
        ),
        "Source": "Quote_Date and Assessment_Date",
        "Feature_Type": "Target",
        "Use_In_Model": "Target only"
    },
    {
        "Feature": "Inactive_180D_Proxy",
        "Definition": (
            "1 when Recency_Days is at least 180; "
            "otherwise 0. Synthetic inactivity proxy."
        ),
        "Source": "Recency_Days",
        "Feature_Type": "Derived proxy",
        "Use_In_Model": "Descriptive segmentation only"
    },
    {
        "Feature": "High_Quote_Value_Flag",
        "Definition": (
            "1 when Total_Quoted_Value is at or above "
            "the 75th percentile of the current dataset"
        ),
        "Source": "Total_Quoted_Value",
        "Feature_Type": "Derived flag",
        "Use_In_Model": "Descriptive segmentation only"
    },
    {
        "Feature": "Recent_Activity_Flag",
        "Definition": (
            "1 when Recency_Days is 90 or fewer; "
            "otherwise 0"
        ),
        "Source": "Recency_Days",
        "Feature_Type": "Derived flag",
        "Use_In_Model": "Descriptive segmentation only"
    },
    {
        "Feature": "Won_Quote_Flag",
        "Definition": (
            "1 when at least one recorded quotation "
            "has Won status; otherwise 0"
        ),
        "Source": "Won_Quote_Count",
        "Feature_Type": "Derived flag",
        "Use_In_Model": "Expansion-candidate rule"
    },
    {
        "Feature": "Customer_Segment",
        "Definition": (
            "Mutually exclusive rule-based segment. "
            "Priority: Inactive Quote, High Quote Value, "
            "Recent Activity, then Standard / Review."
        ),
        "Source": (
            "Inactive_180D_Proxy, High_Quote_Value_Flag, "
            "Recent_Activity_Flag"
        ),
        "Feature_Type": "Descriptive segment",
        "Use_In_Model": "Descriptive analysis only"
    }
])

# ------------------------------------------------------------
# 3. OUTPUT PATHS
# ------------------------------------------------------------

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

target_path = (
    processed_dir / "Day10_Customer_Target_Definitions.csv"
)

feature_path = (
    processed_dir / "Day10_Customer_Feature_Dictionary.csv"
)

# ------------------------------------------------------------
# 4. SAVE OUTPUTS
# ------------------------------------------------------------

target_definitions.to_csv(
    target_path,
    index=False,
    encoding="utf-8"
)

feature_dictionary.to_csv(
    feature_path,
    index=False,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 5. BASIC OUTPUT VALIDATION
# ------------------------------------------------------------

saved_targets = pd.read_csv(target_path)
saved_features = pd.read_csv(feature_path)

assert len(saved_targets) == 3, (
    "Unexpected number of target definitions."
)

assert saved_targets["Target"].is_unique, (
    "Duplicate target names found."
)

assert "Customer_Segment" in saved_targets["Target"].values, (
    "Customer_Segment definition is missing."
)

segment_definition = saved_targets.loc[
    saved_targets["Target"] == "Customer_Segment",
    "Definition"
].iloc[0]

assert "recency" in segment_definition.lower()
assert "inactivity" in segment_definition.lower()

assert not saved_targets.isna().any().any(), (
    "Missing values found in saved target definitions."
)

assert not saved_features.isna().any().any(), (
    "Missing values found in saved feature dictionary."
)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("DAY 10 - CUSTOMER TARGET DEFINITIONS")
print("=" * 70)

print(saved_targets.to_string(index=False))

print("\n" + "=" * 70)
print("DAY 10 - CUSTOMER FEATURE DICTIONARY")
print("=" * 70)

print(saved_features.to_string(index=False))

print("\n" + "=" * 70)
print("OUTPUT VALIDATION")
print("=" * 70)

print(f"Target definitions: {len(saved_targets)}")
print(f"Feature definitions: {len(saved_features)}")
print("Target names unique: PASS")
print("Customer_Segment definition: PASS")
print("Missing-value checks: PASS")

print("\nSaved target definitions:")
print(target_path)

print("\nSaved feature dictionary:")
print(feature_path)

print(
    "\nIMPORTANT: These outputs describe a synthetic demonstration. "
    "Inactivity is not verified churn, quoted value is not collected "
    "revenue, and business/Finance approval remains pending."
)

DAY 10 - CUSTOMER TARGET DEFINITIONS
                 Target                                                                                                                                                                                    Definition                                                                    Required_Fields                      Status                      Approval
          Inactive_180D                                               1 if no quotation is recorded in the 180 days before the assessment date. This is a provisional inactivity proxy, not a verified churn outcome.                                            Company_ID, Quote_Date, Assessment_Date Provisional synthetic proxy Business confirmation pending
       Customer_Segment           Rule-based descriptive grouping using quotation recency, quoted value, and inactivity. Segment priority: Inactive Quote, High Quote Value, Recent Activity, then Standard / Review.               Company_ID, Quote_ID, Quo

In [1]:

import pandas as pd
from pathlib import Path

# ============================================================
# DAY 10 - STEP 1: CUSTOMER DATA READINESS AUDIT
# ============================================================

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

files_to_audit = [
    (
        "data/raw/NexziByte_Final_Scored.csv",
        "Raw prospect dataset"
    ),
    (
        "data/clean/Day3_Prospects_Clean.csv",
        "Clean prospect dataset"
    ),
    (
        "data/synthetic/Day7_Synthetic_Quotation_Dataset.csv",
        "Synthetic quotation dataset"
    ),
    (
        "data/synthetic/Day8_Synthetic_Monthly_Revenue.csv",
        "Synthetic monthly revenue dataset"
    ),
    (
        "data/synthetic/Day9_Synthetic_Project_Risk_Dataset.csv",
        "Synthetic project dataset"
    )
]

# Fields useful for customer intelligence
customer_keywords = [
    "customer", "client", "company", "account",
    "interaction", "activity", "last_activity",
    "contact", "meeting", "support", "renewal",
    "expansion", "invoice", "payment", "revenue",
    "amount", "quote", "accepted_date", "status",
    "service", "industry", "created_date",
    "transaction", "purchase", "churn"
]

audit_rows = []

for relative_path, dataset_type in files_to_audit:
    file_path = project_dir / relative_path

    if not file_path.exists():
        audit_rows.append({
            "File": relative_path,
            "Dataset_Type": dataset_type,
            "File_Exists": False,
            "Rows": None,
            "Columns": None,
            "Customer_Relevant_Columns": "",
            "Data_Status": "File not found"
        })
        continue

    try:
        data = pd.read_csv(file_path)
        columns = list(data.columns)

        matched_columns = [
            col for col in columns
            if any(
                keyword in col.lower()
                for keyword in customer_keywords
            )
        ]

        if "synthetic" in dataset_type.lower():
            status = "Synthetic demonstration only"
        else:
            status = (
                "Existing dataset; customer and "
                "transaction validity requires verification"
            )

        audit_rows.append({
            "File": relative_path,
            "Dataset_Type": dataset_type,
            "File_Exists": True,
            "Rows": len(data),
            "Columns": len(columns),
            "Customer_Relevant_Columns": ", ".join(
                matched_columns
            ),
            "Data_Status": status
        })

    except Exception as error:
        audit_rows.append({
            "File": relative_path,
            "Dataset_Type": dataset_type,
            "File_Exists": True,
            "Rows": None,
            "Columns": None,
            "Customer_Relevant_Columns": "",
            "Data_Status": f"Read error: {error}"
        })

audit_df = pd.DataFrame(audit_rows)

output_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_Customer_Data_Readiness_Audit.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(output_path, index=False)

print("=" * 65)
print("DAY 10 - CUSTOMER DATA READINESS AUDIT")
print("=" * 65)

print(audit_df.to_string(index=False))

print("\nAudit saved to:")
print(output_path)

print(
    "\nNote: keyword matches identify candidate fields only. "
    "They do not prove that customer activity, revenue, or "
    "retention outcomes are available or business-validated."
)


DAY 10 - CUSTOMER DATA READINESS AUDIT
                                                  File                      Dataset_Type  File_Exists  Rows  Columns                                                                                                                                 Customer_Relevant_Columns                                                               Data_Status
                   data/raw/NexziByte_Final_Scored.csv              Raw prospect dataset         True   180       25                          Company Name, Industry / Sub-industry, Products/Services, Target Customers / B2B or B2C, Data Quality Status, Company Size Score Existing dataset; customer and transaction validity requires verification
                   data/clean/Day3_Prospects_Clean.csv            Clean prospect dataset         True   180       28 Company Name, Industry / Sub-industry, Products/Services, Target Customers / B2B or B2C, Data Quality Status, Company Size Score, ETL Company Name Unique 

In [3]:

import pandas as pd
from pathlib import Path

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

quote_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day7_Synthetic_Quotation_Dataset.csv"
)

quotes = pd.read_csv(quote_path)

print("=" * 60)
print("DAY 10 - SYNTHETIC QUOTATION DATA INSPECTION")
print("=" * 60)

print("\nDataset shape:")
print(quotes.shape)

print("\nColumns:")
print(quotes.columns.tolist())

print("\nFirst 10 records:")
display(quotes.head(10))

print("\nData types:")
print(quotes.dtypes)

print("\nMissing values:")
display(quotes.isnull().sum().to_frame("Missing_Count"))

print("\nUnique quotation statuses:")
if "Quote_Status" in quotes.columns:
    print(quotes["Quote_Status"].value_counts(dropna=False))

print("\nQuotation date coverage:")
if "Quote_Date" in quotes.columns:
    quotes["Quote_Date"] = pd.to_datetime(
        quotes["Quote_Date"], errors="coerce"
    )
    print("Earliest quote:", quotes["Quote_Date"].min())
    print("Latest quote:", quotes["Quote_Date"].max())
    print("Invalid dates:", quotes["Quote_Date"].isna().sum())

print("\nUnique companies:")
if "Company_ID" in quotes.columns:
    print(quotes["Company_ID"].nunique())

print("\nSynthetic data only — not verified customer history.")


DAY 10 - SYNTHETIC QUOTATION DATA INSPECTION

Dataset shape:
(300, 12)

Columns:
['Quote_ID', 'Company_ID', 'Quote_Date', 'Quote_Value', 'Discount_Pct', 'Revision_Count', 'Engagement_Count', 'Quote_Status', 'Accepted_Date', 'Loss_Reason', 'Synthetic_Data_Flag', 'Data_Use']

First 10 records:


,Quote_ID,Company_ID,Quote_Date,Quote_Value,Discount_Pct,Revision_Count,Engagement_Count,Quote_Status,Accepted_Date,Loss_Reason,Synthetic_Data_Flag,Data_Use
0,SYN-Q-0001,SYN-C-0001,2026-07-31,58286,20.66,0,9,Won,2026-10-10,NaN,True,Demonstration only; not real business data
1,SYN-Q-0002,SYN-C-0002,2026-07-10,390369,22.40,0,5,Pending,NaN,NaN,True,Demonstration only; not real business data
2,SYN-Q-0003,SYN-C-0003,2026-02-23,332467,3.51,1,10,Lost,NaN,Timing,True,Demonstration only; not real business data
3,SYN-Q-0004,SYN-C-0004,2026-07-21,227856,13.85,4,8,Pending,NaN,NaN,True,Demonstration only; not real business data
4,SYN-Q-0005,SYN-C-0005,2026-01-17,225012,2.71,2,7,Lost,NaN,Requirements mismatch,True,Demonstration only; not real business data
5,SYN-Q-0006,SYN-C-0006,2026-03-10,431420,16.81,3,9,Won,2026-06-05,NaN,True,Demonstration only; not real business data
6,SYN-Q-0007,SYN-C-0007,2026-06-19,56683,7.03,2,5,Lost,NaN,Timing,True,Demonstration only; not real business data
7,SYN-Q-0008,SYN-C-0008,2026-03-14,353224,16.49,0,7,Lost,NaN,Competitor,True,Demonstration only; not real business data
8,SYN-Q-0009,SYN-C-0009,2026-06-25,112712,18.17,1,3,Won,2026-08-18,NaN,True,Demonstration only; not real business data
9,SYN-Q-0010,SYN-C-0010,2026-07-29,60676,19.22,0,10,Won,2026-09-06,NaN,True,Demonstration only; not real business data



Data types:
Quote_ID                object
Company_ID              object
Quote_Date              object
Quote_Value              int64
Discount_Pct           float64
Revision_Count           int64
Engagement_Count         int64
Quote_Status            object
Accepted_Date           object
Loss_Reason             object
Synthetic_Data_Flag       bool
Data_Use                object
dtype: object

Missing values:


,Missing_Count
Quote_ID,0
Company_ID,0
Quote_Date,0
Quote_Value,0
Discount_Pct,0
Revision_Count,0
Engagement_Count,0
Quote_Status,0
Accepted_Date,182
Loss_Reason,163



Unique quotation statuses:
Quote_Status
Lost       137
Won        118
Pending     45
Name: count, dtype: int64

Quotation date coverage:
Earliest quote: 2026-01-02 00:00:00
Latest quote: 2026-08-27 00:00:00
Invalid dates: 0

Unique companies:
300

Synthetic data only — not verified customer history.


In [4]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# DAY 10 - STEP 3: CUSTOMER 360
# Synthetic quotation activity profile
# --------------------------------------------------

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

quote_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day7_Synthetic_Quotation_Dataset.csv"
)

output_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_Customer_360.csv"
)

quotes = pd.read_csv(quote_path)

# Parse dates and validate essential fields.
quotes["Quote_Date"] = pd.to_datetime(
    quotes["Quote_Date"], errors="coerce"
)

quotes["Quote_Value"] = pd.to_numeric(
    quotes["Quote_Value"], errors="coerce"
)

required_columns = [
    "Quote_ID",
    "Company_ID",
    "Quote_Date",
    "Quote_Value",
    "Quote_Status",
]

missing_columns = [
    column for column in required_columns
    if column not in quotes.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

if quotes[["Quote_ID", "Company_ID", "Quote_Date",
           "Quote_Value", "Quote_Status"]].isna().any().any():
    raise ValueError(
        "Essential fields contain missing or invalid values. "
        "Resolve them before building Customer 360."
    )

# Explicit assessment date for this demonstration.
assessment_date = pd.Timestamp("2026-10-19")

# One row per company based on recorded quotation activity.
customer_360 = (
    quotes.groupby("Company_ID", as_index=False)
    .agg(
        Quote_Count=("Quote_ID", "nunique"),
        First_Quote_Date=("Quote_Date", "min"),
        Last_Quote_Date=("Quote_Date", "max"),
        Total_Quoted_Value=("Quote_Value", "sum"),
        Average_Quoted_Value=("Quote_Value", "mean"),
        Won_Quote_Count=(
            "Quote_Status",
            lambda values: (values == "Won").sum()
        ),
        Lost_Quote_Count=(
            "Quote_Status",
            lambda values: (values == "Lost").sum()
        ),
        Pending_Quote_Count=(
            "Quote_Status",
            lambda values: (values == "Pending").sum()
        ),
    )
)

# Recency is measured from the explicit assessment date.
customer_360["Recency_Days"] = (
    assessment_date - customer_360["Last_Quote_Date"]
).dt.days

# Descriptive proxy only; this is NOT a validated churn label.
customer_360["Inactive_180D_Proxy"] = (
    customer_360["Recency_Days"] >= 180
).astype(int)

customer_360["Assessment_Date"] = assessment_date
customer_360["Synthetic_Data_Flag"] = True
customer_360["Data_Use"] = (
    "Synthetic demonstration only; quoted value is not revenue"
)

# Keep the output organized and reproducible.
customer_360 = customer_360.sort_values(
    ["Total_Quoted_Value", "Quote_Count"],
    ascending=[False, False]
).reset_index(drop=True)

output_path.parent.mkdir(parents=True, exist_ok=True)
customer_360.to_csv(output_path, index=False)

# --------------------------------------------------
# Validation and summary
# --------------------------------------------------
print("=" * 60)
print("DAY 10 - CUSTOMER 360 CREATED")
print("=" * 60)

print("\nOutput shape:", customer_360.shape)
print("Unique companies:", customer_360["Company_ID"].nunique())
print("Duplicate company IDs:", customer_360["Company_ID"].duplicated().sum())
print("Missing values:", customer_360.isna().sum().sum())

print("\nQuote count distribution:")
print(customer_360["Quote_Count"].value_counts().sort_index())

print("\nQuotation status totals:")
print("Won:", customer_360["Won_Quote_Count"].sum())
print("Lost:", customer_360["Lost_Quote_Count"].sum())
print("Pending:", customer_360["Pending_Quote_Count"].sum())

print("\nInactivity proxy distribution:")
print(customer_360["Inactive_180D_Proxy"].value_counts().sort_index())

print("\nFirst 10 Customer 360 rows:")
display(customer_360.head(10))

print("\nSaved to:")
print(output_path)

print(
    "\nLIMITATION: This is a synthetic quotation activity profile. "
    "The inactivity flag is a descriptive proxy, not verified churn. "
    "Quoted value is not collected revenue or customer lifetime value."
)


DAY 10 - CUSTOMER 360 CREATED

Output shape: (300, 14)
Unique companies: 300
Duplicate company IDs: 0
Missing values: 0

Quote count distribution:
Quote_Count
1    300
Name: count, dtype: int64

Quotation status totals:
Won: 118
Lost: 137
Pending: 45

Inactivity proxy distribution:
Inactive_180D_Proxy
0    163
1    137
Name: count, dtype: int64

First 10 Customer 360 rows:


,Company_ID,Quote_Count,First_Quote_Date,Last_Quote_Date,Total_Quoted_Value,Average_Quoted_Value,Won_Quote_Count,Lost_Quote_Count,Pending_Quote_Count,Recency_Days,Inactive_180D_Proxy,Assessment_Date,Synthetic_Data_Flag,Data_Use
0,SYN-C-0161,1,2026-02-16,2026-02-16,498112,498112.0,0,1,0,245,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
1,SYN-C-0145,1,2026-04-20,2026-04-20,496443,496443.0,1,0,0,182,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
2,SYN-C-0265,1,2026-06-26,2026-06-26,494997,494997.0,0,0,1,115,0,2026-10-19,True,Synthetic demonstration only; quoted value is ...
3,SYN-C-0141,1,2026-04-16,2026-04-16,491629,491629.0,0,1,0,186,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
4,SYN-C-0012,1,2026-01-06,2026-01-06,488177,488177.0,1,0,0,286,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
5,SYN-C-0046,1,2026-04-08,2026-04-08,485789,485789.0,1,0,0,194,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
6,SYN-C-0208,1,2026-08-16,2026-08-16,485051,485051.0,0,1,0,64,0,2026-10-19,True,Synthetic demonstration only; quoted value is ...
7,SYN-C-0064,1,2026-01-09,2026-01-09,484243,484243.0,1,0,0,283,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
8,SYN-C-0200,1,2026-02-18,2026-02-18,481521,481521.0,0,1,0,243,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...
9,SYN-C-0195,1,2026-03-04,2026-03-04,480451,480451.0,1,0,0,229,1,2026-10-19,True,Synthetic demonstration only; quoted value is ...



Saved to:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day10_Customer_360.csv

LIMITATION: This is a synthetic quotation activity profile. The inactivity flag is a descriptive proxy, not verified churn. Quoted value is not collected revenue or customer lifetime value.


In [5]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# DAY 10 - STEP 4: CUSTOMER SEGMENTATION
# --------------------------------------------------

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

customer_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_Customer_360.csv"
)

segment_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_Customer_Segments.csv"
)

expansion_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_Expansion_Candidates.csv"
)

customers = pd.read_csv(customer_path)

customers["Last_Quote_Date"] = pd.to_datetime(
    customers["Last_Quote_Date"], errors="coerce"
)

customers["Assessment_Date"] = pd.to_datetime(
    customers["Assessment_Date"], errors="coerce"
)

if customers["Total_Quoted_Value"].isna().any():
    raise ValueError("Total_Quoted_Value contains missing values.")

# High-value threshold: 75th percentile of quoted value.
high_value_threshold = customers["Total_Quoted_Value"].quantile(0.75)

customers["High_Quote_Value_Flag"] = (
    customers["Total_Quoted_Value"] >= high_value_threshold
).astype(int)

customers["Recent_Activity_Flag"] = (
    customers["Recency_Days"] <= 90
).astype(int)

# Status is based on the single quotation in this dataset.
customers["Won_Quote_Flag"] = (
    customers["Won_Quote_Count"] > 0
).astype(int)

# Mutually exclusive descriptive segment assignment.
def assign_segment(row):
    if row["Inactive_180D_Proxy"] == 1:
        return "Inactive Quote"
    elif row["High_Quote_Value_Flag"] == 1:
        return "High Quote Value"
    elif row["Recent_Activity_Flag"] == 1:
        return "Recent Activity"
    else:
        return "Standard / Review"

customers["Customer_Segment"] = customers.apply(
    assign_segment, axis=1
)

customers["Segment_Method"] = (
    "Rule-based descriptive segmentation; synthetic data"
)

# Illustrative expansion-candidate heuristic.
# Requires high quoted value, Won status, and recent activity.
customers["Expansion_Candidate_Flag"] = (
    (customers["High_Quote_Value_Flag"] == 1)
    & (customers["Won_Quote_Flag"] == 1)
    & (customers["Recent_Activity_Flag"] == 1)
).astype(int)

customers["Expansion_Candidate_Reason"] = customers[
    "Expansion_Candidate_Flag"
].map({
    1: "High quoted value + Won quotation + recent activity",
    0: "Does not meet the illustrative expansion rule"
})

customers["Data_Use"] = (
    "Synthetic demonstration only; no verified upsell or revenue outcomes"
)

# Save the full segmented Customer 360 dataset.
customers.to_csv(segment_path, index=False)

# Save a separate expansion review list.
expansion_candidates = customers.loc[
    customers["Expansion_Candidate_Flag"] == 1
].copy()

expansion_candidates = expansion_candidates.sort_values(
    "Total_Quoted_Value", ascending=False
)

expansion_candidates.to_csv(expansion_path, index=False)

# --------------------------------------------------
# Validation and summaries
# --------------------------------------------------
print("=" * 60)
print("DAY 10 - CUSTOMER SEGMENTATION RESULTS")
print("=" * 60)

print("\nHigh quoted value threshold:")
print(round(high_value_threshold, 2))

print("\nSegment counts:")
display(
    customers["Customer_Segment"]
    .value_counts()
    .rename_axis("Customer_Segment")
    .reset_index(name="Company_Count")
)

print("\nExpansion candidate count:")
print(len(expansion_candidates))

print("\nExpansion candidates — first 10:")
display(
    expansion_candidates[
        [
            "Company_ID",
            "Quote_Status" if "Quote_Status" in customers.columns
            else "Won_Quote_Count",
            "Total_Quoted_Value",
            "Recency_Days",
            "Customer_Segment",
            "Expansion_Candidate_Reason",
        ]
    ].head(10)
)

print("\nCustomer segment output:")
print(segment_path)

print("\nExpansion candidate output:")
print(expansion_path)

print("\nRows:", len(customers))
print("Unique companies:", customers["Company_ID"].nunique())
print("Duplicate company IDs:", customers["Company_ID"].duplicated().sum())
print("Missing values:", customers.isna().sum().sum())

print(
    "\nLIMITATION: Segments and expansion candidates are heuristic "
    "demonstrations, not validated customer intelligence."
)


DAY 10 - CUSTOMER SEGMENTATION RESULTS

High quoted value threshold:
366968.75

Segment counts:


,Customer_Segment,Company_Count
0,Inactive Quote,137
1,Standard / Review,87
2,Recent Activity,44
3,High Quote Value,32



Expansion candidate count:
2

Expansion candidates — first 10:


,Company_ID,Won_Quote_Count,Total_Quoted_Value,Recency_Days,Customer_Segment,Expansion_Candidate_Reason
22,SYN-C-0179,1,451802,82,High Quote Value,High quoted value + Won quotation + recent act...
55,SYN-C-0278,1,394627,76,High Quote Value,High quoted value + Won quotation + recent act...



Customer segment output:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day10_Customer_Segments.csv

Expansion candidate output:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day10_Expansion_Candidates.csv

Rows: 300
Unique companies: 300
Duplicate company IDs: 0
Missing values: 0

LIMITATION: Segments and expansion candidates are heuristic demonstrations, not validated customer intelligence.


In [6]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# DAY 10 - STEP 5: CHURN MODEL FEASIBILITY AUDIT
# --------------------------------------------------

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

quote_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day7_Synthetic_Quotation_Dataset.csv"
)

output_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_Churn_Model_Feasibility_Audit.csv"
)

quotes = pd.read_csv(quote_path)
quotes["Quote_Date"] = pd.to_datetime(
    quotes["Quote_Date"], errors="coerce"
)

assessment_date = pd.Timestamp("2026-10-19")
future_horizon_days = 180

company_counts = quotes.groupby("Company_ID").agg(
    Quote_Count=("Quote_ID", "nunique"),
    First_Quote_Date=("Quote_Date", "min"),
    Last_Quote_Date=("Quote_Date", "max"),
)

total_companies = len(company_counts)
companies_with_multiple_quotes = int(
    (company_counts["Quote_Count"] > 1).sum()
)

latest_quote_date = quotes["Quote_Date"].max()
earliest_quote_date = quotes["Quote_Date"].min()

# The dataset must contain at least 180 days after an observation cutoff
# to label a complete future 180-day window.
required_coverage_end = assessment_date + pd.Timedelta(
    days=future_horizon_days
)

audit_rows = [
    {
        "Check": "Quotation records",
        "Result": len(quotes),
        "Interpretation": "Synthetic quotation records only",
        "Status": "Informational",
    },
    {
        "Check": "Unique companies",
        "Result": total_companies,
        "Interpretation": "Distinct company identifiers",
        "Status": "Informational",
    },
    {
        "Check": "Companies with multiple quotations",
        "Result": companies_with_multiple_quotes,
        "Interpretation": "Needed to observe repeat quotation activity",
        "Status": (
            "Pass" if companies_with_multiple_quotes > 0
            else "Limitation"
        ),
    },
    {
        "Check": "Earliest quotation date",
        "Result": str(earliest_quote_date.date()),
        "Interpretation": "Beginning of available synthetic history",
        "Status": "Informational",
    },
    {
        "Check": "Latest quotation date",
        "Result": str(latest_quote_date.date()),
        "Interpretation": "End of available synthetic history",
        "Status": "Informational",
    },
    {
        "Check": "Days between latest quote and assessment date",
        "Result": (assessment_date - latest_quote_date).days,
        "Interpretation": "Elapsed time; not proof of churn",
        "Status": "Informational",
    },
    {
        "Check": "Verified churn outcomes available",
        "Result": "No",
        "Interpretation": "No verified retention or churn labels supplied",
        "Status": "Limitation",
    },
    {
        "Check": "Complete future 180-day observation window",
        "Result": "No",
        "Interpretation": (
            f"Assessment date {assessment_date.date()} would require "
            f"coverage through {required_coverage_end.date()}"
        ),
        "Status": "Limitation",
    },
    {
        "Check": "Recency used to construct inactivity target",
        "Result": "Yes",
        "Interpretation": (
            "Do not use the same recency calculation to predict its own target"
        ),
        "Status": "Leakage warning",
    },
]

audit = pd.DataFrame(audit_rows)

output_path.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(output_path, index=False)

print("=" * 60)
print("DAY 10 - CHURN MODEL FEASIBILITY AUDIT")
print("=" * 60)
display(audit)

print("\nSaved to:")
print(output_path)

print(
    "\nDecision: Do not claim a validated predictive churn model "
    "from the current quotation dataset. Use descriptive inactivity "
    "analysis unless suitable longitudinal outcomes become available."
)


DAY 10 - CHURN MODEL FEASIBILITY AUDIT


C:\Users\Sai\AppData\Local\Temp\ipykernel_14104\2429395649.py:48: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  required_coverage_end = assessment_date + pd.Timedelta(


,Check,Result,Interpretation,Status
0,Quotation records,300,Synthetic quotation records only,Informational
1,Unique companies,300,Distinct company identifiers,Informational
2,Companies with multiple quotations,0,Needed to observe repeat quotation activity,Limitation
3,Earliest quotation date,2026-01-02,Beginning of available synthetic history,Informational
4,Latest quotation date,2026-08-27,End of available synthetic history,Informational
5,Days between latest quote and assessment date,53,Elapsed time; not proof of churn,Informational
6,Verified churn outcomes available,No,No verified retention or churn labels supplied,Limitation
7,Complete future 180-day observation window,No,Assessment date 2026-10-19 would require cover...,Limitation
8,Recency used to construct inactivity target,Yes,Do not use the same recency calculation to pre...,Leakage warning



Saved to:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day10_Churn_Model_Feasibility_Audit.csv

Decision: Do not claim a validated predictive churn model from the current quotation dataset. Use descriptive inactivity analysis unless suitable longitudinal outcomes become available.


In [7]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# DAY 10 - STEP 6: CLV FEASIBILITY ASSESSMENT
# --------------------------------------------------

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

output_path = (
    project_dir
    / "data"
    / "processed"
    / "Day10_CLV_Feasibility_Assessment.csv"
)

requirements = [
    {
        "Input": "Customer identifier",
        "Required_For": "Linking customer transactions over time",
        "Available": "Synthetic Company_ID only",
        "Status": "Partial",
        "Action": "Confirm a stable identifier for real customers",
    },
    {
        "Input": "Transaction history",
        "Required_For": "Measuring repeat purchases and customer activity",
        "Available": "One synthetic quotation per company",
        "Status": "Unavailable",
        "Action": "Collect dated customer transaction records",
    },
    {
        "Input": "Collected revenue",
        "Required_For": "Measuring actual customer revenue",
        "Available": "Not available",
        "Status": "Unavailable",
        "Action": "Obtain verified invoices and collected-revenue records",
    },
    {
        "Input": "Delivery or service cost",
        "Required_For": "Estimating customer contribution or profitability",
        "Available": "Not available",
        "Status": "Unavailable",
        "Action": "Obtain customer-level cost data",
    },
    {
        "Input": "Retention or churn history",
        "Required_For": "Estimating future customer relationship duration",
        "Available": "Not available",
        "Status": "Unavailable",
        "Action": "Collect longitudinal retention and churn outcomes",
    },
    {
        "Input": "Observation period",
        "Required_For": "Defining the CLV estimation horizon",
        "Available": "Quotation dates from 2026-01-02 to 2026-08-27",
        "Status": "Insufficient",
        "Action": "Define an approved horizon and collect historical outcomes",
    },
    {
        "Input": "Discount rate",
        "Required_For": "Discounting future customer contribution where appropriate",
        "Available": "Not available",
        "Status": "Unavailable",
        "Action": "Obtain an approved finance assumption if required",
    },
]

assessment = pd.DataFrame(requirements)

output_path.parent.mkdir(parents=True, exist_ok=True)
assessment.to_csv(output_path, index=False)

print("=" * 60)
print("DAY 10 - CLV FEASIBILITY ASSESSMENT")
print("=" * 60)

display(assessment)

print("\nReadiness summary:")
print(assessment["Status"].value_counts())

print("\nSaved to:")
print(output_path)

print(
    "\nDecision: A validated predictive CLV estimate cannot be "
    "calculated from the current data. Quoted value may be used "
    "only as a clearly labelled descriptive proxy."
)


DAY 10 - CLV FEASIBILITY ASSESSMENT


,Input,Required_For,Available,Status,Action
0,Customer identifier,Linking customer transactions over time,Synthetic Company_ID only,Partial,Confirm a stable identifier for real customers
1,Transaction history,Measuring repeat purchases and customer activity,One synthetic quotation per company,Unavailable,Collect dated customer transaction records
2,Collected revenue,Measuring actual customer revenue,Not available,Unavailable,Obtain verified invoices and collected-revenue...
3,Delivery or service cost,Estimating customer contribution or profitability,Not available,Unavailable,Obtain customer-level cost data
4,Retention or churn history,Estimating future customer relationship duration,Not available,Unavailable,Collect longitudinal retention and churn outcomes
5,Observation period,Defining the CLV estimation horizon,Quotation dates from 2026-01-02 to 2026-08-27,Insufficient,Define an approved horizon and collect histori...
6,Discount rate,Discounting future customer contribution where...,Not available,Unavailable,Obtain an approved finance assumption if required



Readiness summary:
Status
Unavailable     5
Partial         1
Insufficient    1
Name: count, dtype: int64

Saved to:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day10_CLV_Feasibility_Assessment.csv

Decision: A validated predictive CLV estimate cannot be calculated from the current data. Quoted value may be used only as a clearly labelled descriptive proxy.


In [8]:

from pathlib import Path
import pandas as pd
from datetime import datetime

# --------------------------------------------------
# DAY 10 - STEP 7: FINAL REPORT AND VALIDATION
# --------------------------------------------------

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

processed_dir = project_dir / "data" / "processed"
synthetic_dir = project_dir / "data" / "synthetic"
reports_dir = project_dir / "reports"

reports_dir.mkdir(parents=True, exist_ok=True)

# Required Day 10 deliverables
required_files = [
    processed_dir / "Day10_Customer_Data_Readiness_Audit.csv",
    processed_dir / "Day10_Customer_Target_Definitions.csv",
    processed_dir / "Day10_Customer_Feature_Dictionary.csv",
    processed_dir / "Day10_Customer_360.csv",
    processed_dir / "Day10_Customer_Segments.csv",
    processed_dir / "Day10_Expansion_Candidates.csv",
    processed_dir / "Day10_Churn_Model_Feasibility_Audit.csv",
    processed_dir / "Day10_CLV_Feasibility_Assessment.csv",
]

# Validate file existence and size.
validation_rows = []

for file_path in required_files:
    exists = file_path.exists()
    size_bytes = file_path.stat().st_size if exists else None

    validation_rows.append({
        "Deliverable": file_path.name,
        "Path": str(file_path),
        "Exists": exists,
        "Size_Bytes": size_bytes,
        "Validation_Status": "PASS" if exists and size_bytes > 0 else "MISSING_OR_EMPTY",
    })

validation = pd.DataFrame(validation_rows)

# Load outputs for content-level validation.
customer_path = processed_dir / "Day10_Customer_360.csv"
segment_path = processed_dir / "Day10_Customer_Segments.csv"
expansion_path = processed_dir / "Day10_Expansion_Candidates.csv"

customers = pd.read_csv(customer_path)
segments = pd.read_csv(segment_path)
expansion = pd.read_csv(expansion_path)

customer_count = len(customers)
unique_company_count = customers["Company_ID"].nunique()
duplicate_company_count = int(customers["Company_ID"].duplicated().sum())
customer_missing_values = int(customers.isna().sum().sum())

segment_counts = (
    segments["Customer_Segment"]
    .value_counts()
    .to_dict()
)

expansion_count = len(expansion)

# Check the expansion candidate rule against saved output.
if expansion_count:
    expansion_rule_valid = bool(
        (
            (expansion["High_Quote_Value_Flag"] == 1)
            & (expansion["Won_Quote_Flag"] == 1)
            & (expansion["Recent_Activity_Flag"] == 1)
        ).all()
    )
else:
    expansion_rule_valid = True

content_checks = {
    "Customer_360_rows_match_unique_companies": (
        customer_count == unique_company_count
    ),
    "No_duplicate_company_IDs": duplicate_company_count == 0,
    "Customer_360_has_no_missing_values": customer_missing_values == 0,
    "Segment_row_count_matches_Customer_360": (
        len(segments) == customer_count
    ),
    "Expansion_candidates_follow_defined_rule": expansion_rule_valid,
}

# Save validation checklist.
validation_path = processed_dir / "Day10_Deliverable_Validation.csv"
validation.to_csv(validation_path, index=False)

# Create the final Markdown report.
report_path = reports_dir / "Day10_Customer_Intelligence_Report.md"

segment_lines = "\n".join(
    f"- **{name}:** {count} companies"
    for name, count in segment_counts.items()
)

missing_files = validation.loc[
    validation["Validation_Status"] != "PASS",
    "Deliverable"
].tolist()

content_check_lines = "\n".join(
    f"- {'PASS' if passed else 'FAIL'} — {check.replace('_', ' ')}"
    for check, passed in content_checks.items()
)

report = f"""# Day 10 — Customer Intelligence Report

## 1. Executive summary

Day 10 developed a synthetic Customer 360 quotation activity profile,
descriptive customer segments, an illustrative expansion-candidate list,
and churn and Customer Lifetime Value (CLV) feasibility assessments.

**Overall status:** Technical demonstration completed; business validation pending.

**Data limitation:** The available quotation dataset is synthetic. Results must
not be represented as actual customer performance, verified churn, collected
revenue, or validated CLV.

## 2. Data readiness

- Source quotation dataset: `data/synthetic/Day7_Synthetic_Quotation_Dataset.csv`
- Quotation records: 300
- Unique companies: 300
- Quotation date range: 2026-01-02 to 2026-08-27
- Won quotations: 118
- Lost quotations: 137
- Pending quotations: 45
- Companies with multiple quotations: 0
- Verified churn outcomes: unavailable
- Actual collected revenue and customer-level cost data: unavailable

The prospect dataset does not establish verified customer transaction history.
Synthetic quotation values are not actual revenue.

## 3. Customer 360

Output: `data/processed/Day10_Customer_360.csv`

- Customer profiles: {customer_count}
- Unique company IDs: {unique_company_count}
- Duplicate company IDs: {duplicate_company_count}
- Missing values: {customer_missing_values}
- Assessment date used for the inactivity proxy: 2026-10-19

The profile contains quotation count, first and last quotation dates,
quoted value summaries, quotation status counts, recency, and a provisional
180-day inactivity proxy.

Every company has one quotation. Therefore, repeat-purchase behavior and
customer retention patterns cannot be measured from this dataset.

The inactivity proxy is descriptive and is not a verified churn outcome.

## 4. Descriptive customer segmentation

Output: `data/processed/Day10_Customer_Segments.csv`

{segment_lines}

Segments are rule-based demonstrations using quoted value and recency.
They have not been validated by business stakeholders.

## 5. Expansion candidates

Output: `data/processed/Day10_Expansion_Candidates.csv`

- Expansion candidates identified: {expansion_count}
- Rule: high quoted value, a Won quotation, and quotation activity within
  the preceding 90 days of the assessment date.

These are candidates for illustrative review only. A Won quotation does not
prove that payment was collected, a service was delivered, or an upsell
opportunity exists.

## 6. Churn feasibility

Output: `data/processed/Day10_Churn_Model_Feasibility_Audit.csv`

A validated predictive churn model was not trained because:

- No company has multiple quotations in the current dataset.
- Verified churn or retention labels are unavailable.
- The current inactivity proxy is derived directly from quotation recency.
- Using that same recency calculation to predict the proxy would introduce
  target leakage.
- The data does not provide a validated future 180-day churn outcome.

The correct next step for a real model is to obtain dated longitudinal
customer activity and an approved outcome definition.

## 7. Customer Lifetime Value feasibility

Output: `data/processed/Day10_CLV_Feasibility_Assessment.csv`

A validated predictive CLV estimate cannot be calculated from the available
data. Missing inputs include actual transaction history, collected revenue,
customer-level costs, retention outcomes, a defined estimation horizon, and
an approved discount-rate assumption where required.

Quoted value may be used as a descriptive monetary proxy only. It must not be
labelled as actual CLV or collected revenue.

## 8. Deliverable validation

{content_check_lines}

Missing or empty files: {", ".join(missing_files) if missing_files else "None"}

Validation checklist: `data/processed/Day10_Deliverable_Validation.csv`

## 9. Business decisions and approvals pending

1. Confirm the definition and business use of customer inactivity.
2. Obtain real customer, transaction, revenue, cost, and retention data.
3. Confirm the business-approved customer segmentation rules.
4. Validate expansion-candidate criteria with the business team.
5. Obtain Finance approval for the CLV definition and assumptions.
6. Reassess churn and CLV modelling after suitable data becomes available.

## 10. Final status

Day 10 technical demonstration deliverables are complete if all required
files and content checks pass. Business validation and production-ready
churn/CLV modelling remain pending.

All synthetic results are for demonstration only and are not evidence of
real business performance.
"""

report_path.write_text(report, encoding="utf-8")

print("=" * 60)
print("DAY 10 - FINAL DELIVERABLE VALIDATION")
print("=" * 60)

display(validation)

print("\nContent checks:")
for check, passed in content_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("\nMissing or empty files:")
print(missing_files if missing_files else "None")

print("\nValidation checklist saved:")
print(validation_path)

print("\nFinal report saved:")
print(report_path)

print("\nDay 10 status:")
if validation["Validation_Status"].eq("PASS").all() and all(content_checks.values()):
    print("TECHNICAL DELIVERABLES COMPLETE — BUSINESS VALIDATION PENDING")
else:
    print("REVIEW REQUIRED — resolve failed checks before sign-off")


DAY 10 - FINAL DELIVERABLE VALIDATION


,Deliverable,Path,Exists,Size_Bytes,Validation_Status
0,Day10_Customer_Data_Readiness_Audit.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,1150,PASS
1,Day10_Customer_Target_Definitions.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,700,PASS
2,Day10_Customer_Feature_Dictionary.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,588,PASS
3,Day10_Customer_360.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,41459,PASS
4,Day10_Customer_Segments.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,81582,PASS
5,Day10_Expansion_Candidates.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,924,PASS
6,Day10_Churn_Model_Feasibility_Audit.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,889,PASS
7,Day10_CLV_Feasibility_Assessment.csv,D:\Nexzibyte Solution\nexzibyte-predictive-bi\...,True,1092,PASS



Content checks:
PASS: Customer_360_rows_match_unique_companies
PASS: No_duplicate_company_IDs
PASS: Customer_360_has_no_missing_values
PASS: Segment_row_count_matches_Customer_360
PASS: Expansion_candidates_follow_defined_rule

Missing or empty files:
None

Validation checklist saved:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day10_Deliverable_Validation.csv

Final report saved:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day10_Customer_Intelligence_Report.md

Day 10 status:
TECHNICAL DELIVERABLES COMPLETE — BUSINESS VALIDATION PENDING
